In [ ]:
from importlib.metadata import version

print("torch version:", version("torch"))

In [ ]:
from dataclasses import dataclass

@dataclass
class ModelArgs:
    n_heads: int
    dim: int
    hidden_dim: int
    dropout: float
    max_seq_len: int
    n_layer: int

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torch
import math

'''多头自注意力核心代码'''
class MultiHeadAttention(nn.Module):

    def __init__(self, args: ModelArgs, is_casual=False):

        # 初始化父类
        super().__init__()

        # 隐藏维度必须被头整除
        assert args.dim % args.n_heads == 0

        # 每个头的维度，等于模型维度除以头的总数。
        self.head_dim = args.dim // args.n_heads

        # 成员变量赋值
        self.n_heads = args.n_heads
        self.is_casual = is_casual

        # Wq, Wk, Wv变换矩阵，shape=[dim, dim], 注意：self.n_heads * self.head_dim=args.dim
        self.wq = nn.Linear(args.dim, self.n_heads * self.head_dim, bias=False)
        self.wk = nn.Linear(args.dim, self.n_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(args.dim, self.n_heads * self.head_dim, bias=False)

        # 输出变换矩阵，shape=[dim, dim]
        self.wo = nn.Linear(self.n_heads * self.head_dim, args.dim, bias=False)

        # 注意力的dropout
        self.attn_dropout = nn.Dropout(args.dropout)

        # 残差的dropout
        self.res_dropout = nn.Dropout(args.dropout)

        # 掩码上三角矩阵，屏蔽未来的token
        #PyTorch 的 .to() 方法只会移动模型内部的“参数（Parameters）”和“缓冲区（Buffers）”。它不会去管普通的 Python 属性。
        if is_casual:
           mask = torch.full((1, 1, args.max_seq_len, args.max_seq_len), float("-inf"))
           mask = torch.triu(mask, diagonal=1)

           # 注册缓冲区
           self.register_buffer("mask", mask)


    def forward(self, q: torch.Tensor, k: torch.Tensor, v: torch.Tensor):

        # [batch_size, seq_len, dim]
        batch_size, seq_len, dim = q.shape

        # 计算Q,K,V，维度为 (batch_size, seq_len, dim) x (dim, dim) -> (batch_size, seq_len, dim)
        Q, K, V = self.wq(q), self.wk(k), self.wv(v)

        # 将 Q、K、V 拆分成多头，shape=(batch_size, seq_len, n_heads, head_dim)
        Q = Q.view(batch_size, seq_len, self.n_heads, self.head_dim)
        K = K.view(batch_size, seq_len, self.n_heads, self.head_dim)
        V = V.view(batch_size, seq_len, self.n_heads, self.head_dim)

        # 交换位置1和位置2，shape=(batch_size, n_heads, seq_len, head_dim)
        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)


        # 注意力计算
        # QK^T / sqrt(d_k)，(batch_size, n_heads, seq_len, head_dim) x (batch_size, n_heads, head_dim, seq_len) -> (batch_size, n_heads, seq_len, seq_len)
        scores = torch.matmul(Q, K.transpose(2, 3)) / math.sqrt(self.head_dim)

        # 计算掩码
        if self.is_casual:
            # 直接相加，-inf叠加的位置就等于mask掉了, max_seq_len可能大于seq_len
            scores = scores + self.mask[:, :, :seq_len, :seq_len]

        # 计算 softmax，shape=(batch_size, n_heads, seq_len, seq_len)
        scores = F.softmax(scores.float(), dim=-1).type_as(Q)

        # 计算dropout，维度不变
        scores = self.attn_dropout(scores)

        # 计算注意力加权输出，V * Score，维度为(batch_size, n_heads, seq_len, seq_len) x (batch_size, n_heads, seq_len, head_dim) -> (batch_size, n_heads, seq_len, head_dim)
        output = torch.matmul(scores, V)

        # 拼接多头注意力结果，(batch_size, n_heads, seq_len, head_dim) => (batch_size, seq_dim, dim)
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)

        # 输出层+残差
        output = self.wo(output)
        output = self.res_dropout(output)

        return output

In [ ]:
class LayerNorm(nn.Module):
    """
    层归一化，用于对最后一个维度进行归一化。

    参数:
        feature_size: 输入特征的维度大小，即归一化的特征维度。
        epsilon: 防止除零的小常数。
    """

    def __init__(self, dim, eps=1e-6):
    	super().__init__()
    	self.gamma = nn.Parameter(torch.ones(dim))  # 可学习缩放参数，初始值为 1
    	self.beta = nn.Parameter(torch.zeros(dim))  # 可学习偏移参数，初始值为 0
    	self.eps = eps

    def forward(self, x):
    	# 计算均值和方差
    	mean = x.mean(-1, keepdim=True) # mean: [batch, max_len, 1]
    	std = x.std(-1, keepdim=True)   # std: [batch, max_len, 1]

    	return self.gamma * (x - mean) / (std + self.eps) + self.beta

In [ ]:
class FFN(nn.Module):
    '''前馈神经网络'''
    def __init__(self, args: ModelArgs):
        super().__init__()
        # 第一个FC，从输入到隐藏层 从T5 开始，很多模型在FFN层都不用偏置了。
        self.w1 = nn.Linear(args.dim, args.hidden_dim, bias=False)

        # 第二个FC，从隐藏层到输入
        self.w2 = nn.Linear(args.hidden_dim, args.dim, bias=False)

        # dropout防止过拟合
        self.dropout = nn.Dropout(args.dropout)

    def forward(self, x):
        # 前向传播函数
        # 首先，输入x通过第一层线性变换和RELU激活函数
        # 最后，通过第二层线性变换和dropout层
        return self.dropout(self.w2(F.relu(self.w1(x))))

In [ ]:
class EncoderLayer(nn.Module):
    '''Encoder层'''
    def __init__(self, args):
        super().__init__()

        # 两个 LayerNorm，分别在 Attention 之前和 FFN 之前
        self.attention_norm = LayerNorm(args.dim)

        # Encoder不需要掩码
        self.attention = MultiHeadAttention(args, is_casual=False)

        self.fnn_norm = LayerNorm(args.dim)

        self.feed_forward = FFN(args)

    def forward(self, x):
        # 层归一化
        norm_x = self.attention_norm(x)

        # 多头自注意力
        h = x + self.attention.forward(norm_x, norm_x, norm_x)

        # 前馈神经网络
        out = h + self.feed_forward.forward(self.fnn_norm(h))

        return out

In [ ]:
class Encoder(nn.Module):
    '''Encoder 块'''
    def __init__(self, args:ModelArgs):
        super(Encoder, self).__init__()
        #  N 个 Encoder Layer叠加
        self.layers = nn.ModuleList([EncoderLayer(args) for _ in range(args.n_layer)])
        self.norm = LayerNorm(args.dim)

    def forward(self, x):
        "分别通过 N 层 Encoder Layer"
        for layer in self.layers:
            x = layer(x)
        return self.norm(x)

解码器

In [ ]:
#解码层的核心实现
class DecoderLayer(nn.Module):
    def __init__(self,args:ModelArgs):
        super().__init__()

        #Mask attention的归一化
        self.attention_mask_norm = LayerNorm(args.dim)
        #Self attention的归一化
        self.attention_norm_ = LayerNorm(args.dim)
        #FFN的归一化
        self.ffn_norm = LayerNorm(args.dim)

        #Mask self attention
        self.mask_attention = MultiHeadAttention(args,is_casual=True)

        #Cross Attention
        self.cross_attention = MultiHeadAttention(args,is_casual=False)

        #FFN
        self.feed_forward = FFN(args)

    # 对编码器特征的压缩，传入到Cross Attention中充当k v
    def forward(self,x,enc_out):
        #掩码注意力层归一化
        norm_x = self.attention_mask_norm(x)

        #掩码注意力 + 残差
        x = x + self.mask_attention(norm_x, norm_x, norm_x)

        #交叉注意力层归一化 - 归一化更新后的x，而非旧的norm_x
        norm_x = self.attention_norm_(x)

        # 交叉多头注意力 + 残差
        h = x +self.cross_attention.forward(norm_x, enc_out,enc_out)

        #FFN + 残差
        out = h +self.feed_forward.forward(self.ffn_norm(h))
        return out

In [ ]:
#多层解码层的叠加
class Decoder(nn.Module):
    def __init__(self,args):
        super().__init__()

        #一个Decoder由多个DecoderLayer组成
        self.layers = nn.ModuleList([DecoderLayer(args) for _ in range(args.n_layer)])

        #最后输出还会有一个归一化层
        self.norm = LayerNorm(args.dim)

    def forward(self,x,enc_out):
        #给每一层输入和enc_out做解码
        for layer in self.layers:
            x = layer(x,enc_out)
        #最后的输出也要归一化
        return self.norm(x)

Case1

In [ ]:
args = ModelArgs(n_heads=8, dim=768, hidden_dim=768*4, dropout=0.1, max_seq_len=512, n_layers=6)
print(args)

batch_size = 10

# 定义输入
x = torch.randn(batch_size, args.max_seq_len, args.dim)

# Encoder
encoder = Encoder(args)

# Decoder
decoder = Decoder(args)

# 编码
encoder_output = encoder(x)
decoder_output = decoder(x, encoder_output)
print("decoder output shape: ", decoder_output.shape)
print("decoder output: ", decoder_output)